<a href="https://colab.research.google.com/github/Chaitralikore/Remote-Sensing/blob/main/Pansharpening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Prepare your Layers
Before running the tool, you need two specific things in your Layers Panel:

The RGB Composite: Create a "Virtual Raster" or "Merge" of Bands 4 (Red), 3 (Green), and 2 (Blue). This is your "Low-Resolution Color."

The Panchromatic Band: Add Band 8 (B8). This is your "High-Resolution Black & White" (15m resolution).


---



Step 2: Build the Virtual Raster (Spectral Composite)
Go to Raster > Miscellaneous > Build Virtual Raster.

Crucial: You MUST check the box """"Place each input file into a separate band

Input Layers: Select B4, B3, and B2 (ensure they are in that specific order).

Check the box "Place each input file into a separate band."

Click Run and save this as RGB_Composite.


---



Step 3: Perform Pansharpening (GDAL)
Now we combine the 30m color image with the 15m panchromatic image.

Open the Processing Toolbox (Ctrl+Alt+T).

Search for "Pansharpening" and select GDAL > Raster miscellaneous > Pansharpening.

Spectral Dataset: Select your RGB_Composite (the virtual raster from Step 2).

Panchromatic Dataset: Select Band 8 (B8).

Resampling Algorithm: Change this to Cubic or Lanczos for the best visual quality.

Pansharpened (Output): Click the three dots [...] and save as Exp6_Pansharpened.tif.

Click Run.


---



Step 4: Compare the Results (The "Why")
To see the difference, zoom in closely on a road, a building, or a coastline:

Toggle the RGB Composite: It will look slightly "blocky" or blurred because it is 30m resolution.

Toggle the Pansharpened Layer: It should look significantly sharper and more detailed, almost like a higher-quality camera took the photo.

Summary for your Report Procedure:
Procedure:

Data Preparation: Loaded Landsat 8 multispectral bands (B4, B3, B2) and the panchromatic band (B8).

Spectral Composition: Created a 30m resolution virtual raster to act as the multispectral input.

Algorithm Execution: Applied the GDAL Pansharpening algorithm, using the 15m panchromatic band to provide spatial detail and the virtual raster to provide spectral information.

Observation: Noticed a significant increase in spatial resolution while maintaining the original color characteristics of the RGB composite.

https://earthengine.google.com/
Click 'Get Started', Enter your personal Gmail account, Click on Register project, Select unpaid usage and Project type as Academia and research, Click Create a new Google cloud project and Continue with registration
Remember your Project-ID for using in the program)


---

pansharpening-495423


---

enable: https://console.cloud.google.com/apis/library/earthengine.googleapis.com?project=pansharpening-495423

In [3]:
#Part A : HSV Transformation
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='pansharpening-495423')

In [4]:
#load a landsat 8 top-of-atmosphere reectance image
image=ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318')

#CONVERT the RGB bands to HSV color space.
hsv=image.select(['B4','B3','B2']).rgbToHsv()

#swap in panchromatic band and convert back to RGB
sharpened =ee.Image.cat(
    [hsv.select('hue'),
    hsv.select('saturation'),
    image.select('B8')]
).hsvToRgb()

In [5]:
#Define a map
map_sharpened = geemap.Map(center=[37.76664, -122.44289],zoom=13)
#Add the image layer to the map and display it
map_sharpened.add_layer(
    image,
    {
        'bands':['B4','B3', 'B2'],
        'min': 0,
        'max':0.25,
        'gamma':[1.1,1.1,1],
    },
    'rgb',
)
map_sharpened.add_layer(
    sharpened,
    {
        'min': 0,
        'max':0.25,
        'gamma':[1.3,1.3,1.3],
    },
    'pan-sharpened',
)
display(map_sharpened)


Map(center=[37.76664, -122.44289], controls=(WidgetControl(options=['position', 'transparent_bg'], position='t…

In [6]:
#Part B:Brovey Transformation
image=ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318')
rgb = image.select(['B4','B3','B2'])
pan = image.select('B8')
sum_rgb = rgb.reduce(ee.Reducer.sum())
brovey = rgb.divide(sum_rgb).multiply(pan)
map_brovey_sharpened = geemap.Map(center=[37.76664, -122.44289],zoom=13)
map_brovey_sharpened.add_layer(
    image,
{
        'bands':['B4','B3', 'B2'],
        'min': 0,
        'max':0.25,
        'gamma':[1.1,1.1,1],
    },
    'original rgb',
)
map_sharpened.add_layer(
    brovey,
    {
        'min': 0,
        'max':0.3,
        'gamma':[1.3,1.3,1.3],
    },
    'brovey pan-sharpened',
)
display(map_brovey_sharpened)

Map(center=[37.76664, -122.44289], controls=(WidgetControl(options=['position', 'transparent_bg'], position='t…

Open Tool: Go to Raster > Miscellaneous > Build Virtual Raster...

Input Layers: Click the three dots [...] and select LC09_B4 (Red), LC09_B3 (Green), and LC09_B2 (Blue).

Crucial Setting: Check the box that says "Place each input file into a separate band".

If you miss this, you will only have "Band 1" later, and your image will stay grayscale.

Resolution: Set Resolution to Highest.

Run: Save this as Spectral_Composite.vrt and click Run.

Phase 2: Run the Pansharpening Algorithm
This step injects the 15m spatial detail from Band 8 into the 30m color from your composite.

Open Tool: In the Processing Toolbox on the right, search for "Pansharpening" and double-click GDAL > Raster miscellaneous > Pansharpening.

Spectral Dataset: Select the Spectral_Composite you just created.

Panchromatic Dataset: Select LC09_B8 (the 15m resolution band).

Resampling: Change this to Cubic for the smoothest high-quality result.

Output: Save the file as Exp6_Pansharpened_Final.tif.

Run: Click Run.

Phase 3: Set True Color Symbology
Because the output is a new file, QGIS may display it in black and white by default.

Properties: Right-click your new Pansharpened layer and select Properties.

Symbology:

Render type: Change to Multiband color.

Band Mapping: Now that you used the "Separate Band" option in Phase 1, you should see all three bands:

Red band: Band 1 (B4)

Green band: Band 2 (B3)

Blue band: Band 3 (B2)

Enhancement: Set Contrast enhancement to Stretch to MinMax.

Apply: Click Apply and OK.

# **q**

EXP 6 Pan-sharpening Algorithm
1. Image Transformation Techniques in Remote Sensing
● Linear Transformation:
○ Principal Component Analysis (PCA): Transforms the original bands into uncorrelated components.
○ Intensity-Hue-Saturation (IHS): Separates image into intensity, hue, and saturation, often used in pansharpening.
● Non-Linear Transformation:
○ Brovey Transformation: A color-space-based method that uses the ratio of individual bands to the sum of the RGB bands and multiplies it with the panchromatic band.
○ Histogram Equalization: Enhances contrast by redistributing pixel intensity values.
● Fourier Transform:
Frequency Domain: Used for filtering and noise reduction by transforming spatial domain data
into frequency components.
2. Pansharpening in Remote Sensing
● Definition:
○ Pansharpening refers to the process of combining high-resolution panchromatic images with lower-resolution multispectral images to create a high-resolution color image. ○ It enhances the spatial resolution of multispectral imagery while preserving spectral information.
● Applications:
○ Urban Planning: High-resolution images are used for detailed urban mapping. ○ Agriculture: Helps in analyzing crop health and land management.
○ Disaster Management: Improved imagery is critical for assessing post-disaster


---
. Pansharpening Methods
● IHS (Intensity-Hue-Saturation) Transformation:
○ Method:
■ Convert RGB bands to HSV (Hue, Saturation, Value) color space.
■ Replace the intensity component (Value) with the high-resolution panchromatic image.
■ Convert back to RGB to create a sharp image.
○ Advantages: It maintains the color integrity of the image while enhancing spatial resolution.
● Brovey Transformation:
○ Method:
■ The individual RGB bands are divided by their sum and then multiplied by the panchromatic band.
■ The result is a sharper multispectral image.
○ Advantages: Simple to implement and effective for enhancing sharpness, especially when using low-resolution multispectral and high-resolution panchromatic images.


---

Relevant Theory for the Code
● HSV Transformation:
○ HSV color space: Separates chromatic content (Hue) from intensity (Value) to enhance spatial resolution without distorting color.
○ IHS Pansharpening: Intensity component replaced with the high-resolution panchromatic band while maintaining color fidelity.
● Brovey Transformation:
○ A multiplicative method that sharpens the image by using the relative proportions of the RGB bands, making it effective for enhancing spatial resolution.
○ Use Case: Particularly effective when a high-resolution panchromatic band is available, and the goal is to improve the sharpness of multispectral bands.
Expected Viva Questions
1. What is pansharpening, and why is it important in remote sensing?
○ Pansharpening is the process of combining a high-resolution panchromatic image (typically black and white) with a low-resolution multispectral image (colored) to create a high-resolution color image.
○ It is important in remote sensing because it improves the spatial resolution of multispectral images while retaining their spectral integrity, making them more useful for detailed analysis in applications such as urban planning, agriculture, and disaster monitoring.
2. Explain the IHS pansharpening method in detail.
○ IHS (Intensity-Hue-Saturation) pansharpening involves transforming the multispectral image into the HSV color space.
○ Steps:
1. Convert the RGB bands into HSV.
2. Replace the Intensity (Value) component of the HSV image with the
high-resolution panchromatic image.
3. Convert the modified HSV image back into RGB to generate the final sharpened image.
○ Advantage: This method helps preserve the original color information while enhancing the spatial resolution using the panchromatic band.
3. How does Brovey transformation differ from IHS?
○ Brovey Transformation is a multiplicative method, whereas IHS is an additive method.
○ Brovey Transformation involves dividing the RGB bands by their sum and then multiplying by the panchromatic band to sharpen the image. This emphasizes the spatial resolution enhancement without distorting the spectral characteristics too much.
○ IHS focuses on replacing the intensity component of the HSV model with the panchromatic image, making it less computationally expensive compared to the Brovey method, which requires handling the entire RGB sum.
○ Brovey is often simpler to implement, while IHS can better maintain the color integrity.
4. What is the role of the panchromatic band in pansharpening?
○ The panchromatic band is a high-resolution grayscale image that captures fine spatial details. In pansharpening, it is used to enhance the spatial resolution of multispectral images by replacing the low-resolution intensity component of the color image.
○ The panchromatic band’s high spatial resolution helps sharpen the features in the multispectral image, resulting in a clearer, more detailed output image.
5. How do you implement pansharpening in QGIS?
○ In QGIS, the process involves the following steps:
1. Download the multispectral and panchromatic bands (e.g., from Landsat). 2. Create a composite of the bands you wish to sharpen (e.g., bands B4, B3, B2 for RGB).
3. Use GDAL's Raster Miscellaneous > Pansharpening option to merge the spectral bands with the panchromatic band.
4. Select the pansharpening method (e.g., IHS or Brovey) in QGIS and run the process to generate a high-resolution image.
6. Explain the purpose of the rgbToHsv() and hsvToRgb() methods in the code. ○ rgbToHsv(): Converts the input RGB bands to the HSV color space (Hue, Saturation, Value). This is done to separate the color information (Hue and Saturation) from the intensity information (Value), allowing us to replace the intensity with the high-resolution panchromatic image.
○ hsvToRgb(): Converts the HSV image back to RGB after replacing the intensity component with the panchromatic image. This allows the result to be visualized in a color format.
7. What is the significance of the reduce(ee.Reducer.sum()) function in the Brovey transformation?
○ The reduce(ee.Reducer.sum()) function sums up the values of the RGB bands. This sum is then used as a normalization factor to divide each individual RGB band before multiplying by the panchromatic band.
○ It ensures that the transformed RGB values are proportionally scaled based on their total, thus preventing excessive intensity values and making the transformation more effective.
